# 02 - Tiền Xử Lý & Feature Engineering

**Mục tiêu:**
- Xử lý missing values
- Xử lý outlier
- Encoding biến phân loại
- Rời rạc hoá cho Apriori
- Lưu dữ liệu đã xử lý

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 2.1 Load dữ liệu thô

In [ ]:
from src.data.loader import load_raw_data, load_params

params = load_params()
df = load_raw_data()
print(f"Shape ban đầu: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

## 2.2 Pipeline tiền xử lý

Các bước:
1. Drop cột không cần (`id`, `dataset`)
2. Tạo binary target (num → target)
3. Xử lý missing values (fill median/mode)
4. Encode biến phân loại (Label Encoding)
5. Handle outlier (IQR clipping)

In [ ]:
from src.data.cleaner import full_cleaning_pipeline

df_clean = full_cleaning_pipeline(
    df,
    drop_cols=params['preprocessing']['drop_cols'],
    missing_strategy=params['preprocessing']['missing_strategy'],
    missing_threshold=params['preprocessing']['missing_threshold'],
    encode_method=params['preprocessing']['encode_method'],
    handle_outliers=params['preprocessing']['handle_outliers'],
)

In [ ]:
# Kiểm tra kết quả
print(f"Shape sau tiền xử lý: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df_clean['target'].value_counts())
df_clean.head()

## 2.3 Kiểm tra Duplicates

In [ ]:
# Kiểm tra duplicate
dupes = df_clean.duplicated().sum()
print(f"Số dòng trùng lặp: {dupes}")
if dupes > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Đã xoá {dupes} dòng. Shape mới: {df_clean.shape}")
else:
    print("Không có dòng trùng lặp")

## 2.4 Thống kê trước vs sau tiền xử lý

In [ ]:
# So sánh trước - sau chi tiết
comparison = pd.DataFrame({
    'Before': [df.shape[0], df.shape[1], df.isnull().sum().sum(), df.duplicated().sum()],
    'After': [df_clean.shape[0], df_clean.shape[1], df_clean.isnull().sum().sum(), df_clean.duplicated().sum()]
}, index=['Rows', 'Columns', 'Missing Values', 'Duplicates'])
print("=== THỐNG KÊ TRƯỚC vs SAU TIỀN XỬ LÝ ===")
comparison

In [ ]:
# So sánh mean/std trước-sau cho biến số
before_stats = df.select_dtypes(include='number').describe().loc[['mean','std']].T
after_stats = df_clean.select_dtypes(include='number').describe().loc[['mean','std']].T
compare_stats = pd.DataFrame({
    'Before Mean': before_stats['mean'],
    'After Mean': after_stats['mean'],
    'Before Std': before_stats['std'],
    'After Std': after_stats['std']
}).dropna().round(2)
print("=== SO SÁNH MEAN/STD TRƯỚC-SAU ===")
compare_stats

In [ ]:
# Phân phối các biến số sau tiền xử lý
from src.visualization.plots import plot_numeric_distributions
fig = plot_numeric_distributions(df_clean, save=False)
plt.suptitle('Distributions After Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.4 Feature Engineering - Rời rạc hoá cho Apriori

In [ ]:
from src.features.builder import discretize_for_apriori

# Dùng dữ liệu gốc (chưa encode) cho rời rạc hoá
df_for_disc = df.drop(columns=['id', 'dataset']).copy()
df_for_disc['target'] = (df_for_disc['num'] > 0).astype(int)

df_disc = discretize_for_apriori(df_for_disc)
print(f"\nShape: {df_disc.shape}")
df_disc.head()

## 2.5 Chuẩn bị X, y cho Modeling

In [ ]:
from src.features.builder import select_features_for_modeling

X, y = select_features_for_modeling(df_clean)

## 2.6 Lưu dữ liệu đã xử lý

In [ ]:
import os
os.makedirs(os.path.dirname(params['paths']['processed_data']), exist_ok=True)

# Lưu processed data
df_clean.to_csv(params['paths']['processed_data'], index=False)
print(f"💾 Saved: {params['paths']['processed_data']}")

# Lưu discretized data
df_disc.to_csv(params['paths']['discretized_data'], index=False)
print(f"💾 Saved: {params['paths']['discretized_data']}")

## 2.7 Nhận xét

- **Missing values:** Xử lý bằng median (số) và mode (phân loại)
- **Encoding:** Label Encoding cho 5 biến phân loại
- **Outlier:** Chỉ clip 4 biến liên tục thật sự (age, trestbps, chol, thalch, oldpeak)
- **Rời rạc hoá:** Tạo được binary features cho Apriori dựa trên ngưỡng y tế
- Dữ liệu sẵn sàng cho bước Mining và Modeling